---

# The Notebook Introduction Cell

---

# Comprehensive Time-Series Forecasting Pipeline
## An Architectural Deep Dive from Feedforward Baselines to Gated Recurrence

Welcome to this interactive research notebook. The purpose of this project is to implement, evaluate, and benchmark multiple deep learning configurations on a univariate time-series sequence (using the classic *Air Passengers* tracking dataset). 

### Key Objectives
1. **Modular Engineering:** Transition away from dense, messy code blocks by keeping structural compilation patterns abstracted inside an isolated `models/` repository directory.
2. **Structural Comparison:** Analyze how various neural network paradigms—ranging from standard spatial MLPs and spatial 1D CNN feature extractors to deep recurrent LSTMs—manipulate inputs.
3. **Hyperparameter Grid Exploration:** Lay down the computational architecture needed to sweep across sliding window configurations, layer counts, and tuning metrics.

---

### The Core Challenge: Transforming Sequences to Supervised Data
Deep learning architectures require distinct feature metrics ($X$) mapped to precise target arrays ($y$). To achieve this with a singular continuous stream of numbers, we deploy the **Sliding Window Method**. 

A predefined constraint variable—**Lookback Window ($n\_steps$)**—shifts across the sequence chronologically, packaging previous observation steps into independent training samples to forecast the next immediate incremental value.

## 1. MLP (Multi-Layer Perception)

### 1.1 Overview
A Multilayer Perceptron (MLP) is a foundational class of feedforward artificial neural networks. It consists of the following:
- input layer
- one or more hidden layers utilizing non-linear activation functions (typically ReLU)
- and an output layer.

In standard machine learning, MLPs are designed for tabular data, where each row represents an independent sample, and each column represents a distinct features variable. They have no inherent concept of time, sequence, or temporal order. They treat the value at time step $t-1$ and time step $t-3$ as entirely independent inputs, completely unaware of which one occurred first.

### 1.2 The Supervised Learning Strategy

To use an MLP for univariate time-series forecasting, we must artificially restructure our sequential data into a supervised learning framework using the sliding window (lookback) technique. If we have a lookback window size of $n\_steps = 3$, we map the sequence as follows:

| Input Sequence (X) | Target Label (y) |
|--------------------|------------------|
| $[x_1, x_2, x_3]$  | $x_4$            |
| $[x_2, x_3, x_4]$  | $x_5$            |
| $[x_3, x_4, x_5]$  | $x_6$            |

### 1.3 Tensor Transformations and Dimensions
When using standard deep learning frameworks like Keras, time-series data preparation pipelines generate a 3D input tensor to maintain compatibility across sequential models (like RNNs and LSTMs).

As professor noted in class, the 3d Input Shape is constructed by:

Input Shape: $$\text{Shape} = (\text{Samples}, \text{Time Steps}, \text{Features})$$

- Samples: The data viewpoints in the sliding window
- Time_steps ($n\_steps$): The lookback window size
- Features ($n\_features$): The number of variables or parallel series. In the case of univariate forecasting, this is set to 1.

### 1.4 The Flattening Mechanism:

Because an MLP cannot process a 3D temporal structure, its first layer must be a Flatten Layer. This layer collapses the spatial dimensions $(Time Steps \times Features)$ into a single 1D feature vector for each sample.

Therefore, say that we have the following:

Input: (Samples, 3, 2)

The result of the flattening mechanism would transform this into: 

Input: (Samples, 6)

This tells the hidden layers to process the inputs as 6 separate independent tabular parameters rather than a sequence.

### 1.4 Architectural Structure

Below is the conceptual structure of the architecture we will import from our repository:
- Input Shape Configuration: Receives the 3D tensor (n_steps, n_features).
- Flatten Layer: Linearizes the window inputs into a shape of (n_steps * n_features).
- Dense Hidden Layer: 100 fully-connected nodes utilizing the ReLU (Rectified Linear Unit) activation function to capture non-linear patterns within the window.
- Dense Output Layer: A single node with a linear activation function to yield a continuous value forecast for time step $t$.

In [1]:
from models.mlp import build_mlp_model

# Define your input window constraints
n_steps = 3
n_features = 1

# Instantiate the architecture
model = build_mlp_model(n_steps, n_features)
model.summary()

c:\Users\Lanie\Documents\Code\Keras-Time-Series-Architectures\.venv_tf\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 3)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 501 (1.96 KB)

 Trainable params: 501 (1.96 KB)

 Non-trainable params: 0 (0.00 B)

# 2. Multi-Head MLP

### 2.1 Overview
A standard MLP expects a single flattened input vector representing a single lookback window. However, when dealing with multivariate time series—where multiple parallel sequences (e.g., ambient temperature, relative humidity, and power consumption) are tracked simultaneously—a single wide matrix can muddy the feature extraction process. 

A **Multi-Head MLP** resolves this by deploying the **Keras Functional API** to instantiate a dedicated, isolated input and interpretation "head" for each individual sequence. This allows the model to extract distinct patterns from each variable independently before merging their feature representations to execute a forecast.

### 2.2 Tensor Transformations and Dimensions
Because each input series receives its own dedicated head, the data is not passed as a uniform 3D or wide 2D tensor. Instead, the model graph expects a **list of independent 2D arrays**, with one array per head.

$$\text{Input Sub-Shapes: } [(\text{Samples}, n\_steps)_1, (\text{Samples}, n\_steps)_2, \dots, (\text{Samples}, n\_steps)_n]$$

The multi-stage feature aggregation pipeline flows as follows:
1. **Isolated Processing:** Each head passes its tracking slice through a dedicated fully connected layer, mapping $(\text{Samples}, n\_steps) \rightarrow (\text{Samples}, 30)$.
2. **Feature Fusion (Concatenate):** The independent hidden representations are chained together along the feature axis.
3. **Combined Bottleneck:** The fused vector is sent to a downstream master network to capture cross-variable interactions.

$$\text{Fusion Tensor Map: } [(\text{Samples}, 30) \times n\_heads] \xrightarrow{\text{Concatenate}} (\text{Samples}, 30 \times n\_heads) \xrightarrow{\text{Dense(50)}} (\text{Samples}, 50)$$

### 2.3 Architectural Structure
The pipeline imported from our multi-head framework implements:
- **Functional Input Array:** A list of $N$ `Input` objects matching `n_heads`, each mapping a historical lookback window of `n_steps`.
- **Per-Head Dense Layer:** 30 nodes utilizing the ReLU activation function per stream to isolate initial feature curves.
- **Concatenation Layer:** Fuses the outputs of all parallel heads into a unified tensor map.
- **Combined Dense Interpretation Layer:** 50 nodes with ReLU activation to learn cross-feature correlations.
- **Dense Output Layer:** A single node acting as a linear regressor to output the continuous value forecast for time step $t+1$.

# 4. Recurrent Neural Network (RNN) 

### 4.1 Overview
Unlike feedforward networks (like MLPs) that process all inputs simultaneously and independently, a Recurrent Neural Network (RNN) processes sequences step-by-step. It maintains an internal hidden state ($h_t$) that acts as a working memory, carrying information from previous time steps forward to influence the processing of the current time step.

At any given time step $t$, the RNN cell takes two inputs:
- The current sequential data point ($x_t$).
- The hidden state from the previous step ($h_{t-1}$).

**The Vanishing Gradient Problem**

While powerful in theory, Vanilla RNNs (often called SimpleRNNs) struggle with long time-series sequences. During backpropagation, gradients are calculated by multiplying weights across every time step (Backpropagation Through Time, or BPTT). If these weights are small, the gradient shrinks exponentially as it travels back to early time steps, causing the network to "forget" long-term historical context.

## 4.2 Tensor Transformations and Dimensions

Because RNNs natively process temporal sequences, the input data is not flatten. The network preserves the explicit timeline.

Input Shape: $$\text{Shape} = (\text{Samples}, \text{Time Steps}, \text{Features})$$

- Time Steps ($n\_steps$): Explored sequentially by the recurrent cells. If $n\_steps=3$, the RNN cell internal loop executes exactly three times per sample.
- Features ($n\_features$): The number of concurrent dimensions entering the cell at each step ($1$ for univariate).

Hidden State Transition:$$\text{Input: } (\text{Samples}, \text{Time Steps}, \text{Features}) \xrightarrow{\text{SimpleRNN(units=50)}} \text{Output: } (\text{Samples}, 50)$$By default, the SimpleRNN layer processes the entire timeline but only outputs the final hidden state vector ($h_{\text{final}}$) calculated at the very last time step. This vector represents a compressed temporal summary of the entire window, which is then passed to a Dense layer for prediction.

### 4.3 Architectural Structure

The pipeline we will import features:
- SimpleRNN Layer: 50 recurrent units utilizing the Hyperbolic Tangent (tanh) activation function (the standard defaults for structural stability in recurrent loops).
- Dense Output Layer: A single node to maps the final hidden state vector into a continuous numeric forecast for time step $t$.

In [2]:
# Import the model architecture from the models directory
from models.rnn import build_rnn_model

# Configuration variables
n_steps = 3
n_features = 1

# Instantiate and build the model
rnn_model = build_rnn_model(n_steps, n_features)

# Display the structural summary and parameter counts
rnn_model.summary()

c:\Users\Lanie\Documents\Code\Keras-Time-Series-Architectures\.venv_tf\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 50)             │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,651 (10.36 KB)

 Trainable params: 2,651 (10.36 KB)

 Non-trainable params: 0 (0.00 B)

# 5. 1D Convolutional Neural Network (CNN) for Time Series

### 5.1 Overview
While 2D Convolutional Neural Networks are the standard for spatial image processing, 1D CNNs are highly effective for sequential and time-series analysis. Instead of scanning an image from top-to-bottom and left-to-right, a 1D CNN slides a convolutional kernel (or filter) across a single dimension: the timeline.

As the kernel moves across the lookback window, it performs matrix multiplications to extract highly localized, position-invariant features. This makes 1D CNNs exceptional at detecting transient, structural patterns—such as sudden momentum shifts, structural breaks, or seasonal spikes—independent of exactly where they occur in the lookback window.

Advantages of CNNs over RNNs:
- Parallelization: Unlike RNNs, which must calculate steps sequentially ($t_1 \rightarrow t_2 \rightarrow t_3$), a CNN processes overlapping window segments simultaneously. This makes them significantly faster to train.

- No Vanishing Gradients: Because the architecture doesn't rely on a deep recurrent feedback loop over long horizons, it avoids the vanishing gradient issues typical of vanilla recurrent units.

### 5.2 Tensor Transformations and Dimensions

A 1D CNN preserves the temporal sequence but alters the feature dimensionality as the tensor moves through the feature extraction pipeline.

Input Shape:$$\text{Shape} = (\text{Samples}, \text{Time Steps}, \text{Features})$$

The Feature Extraction Pipeline:
1. Convolution (Conv1D): Slides $N$ filters across the timeline. If your input is (Samples, 3, 1) and you apply 64 filters with a kernel size of 2, the layer learns 64 distinct localized patterns, reshaping the output tensor to (Samples, 2, 64).
2. Pooling (MaxPooling1D): Sub-samples the sequence by extracting the maximum value across a small sliding window (e.g., pool_size=2). This condenses the timeline down to the most prominent feature responses.
3. Flattening (Flatten): Flattens the remaining temporal feature maps into a single 1D vector per sample, preparing it for the fully connected decision layers.$$\text{Input: } (\text{Samples}, 3, 1) \xrightarrow{\text{Conv1D + MaxPool}} (\text{Samples}, 1, 64) \xrightarrow{\text{Flatten}} \text{Output: } (\text{Samples}, 64)$$

### 5.3 Architectural Structure

The pipeline we will import features:

- Conv1D Layer: 64 filters with a kernel size of 2, utilizing the ReLU activation function to introduce non-linearity.
- MaxPooling1D Layer: A pooling window of size 2 to downsample the feature maps.
- Flatten Layer: Collapses the multi-dimensional feature map into a flat vector.
- Dense Hidden Layer: 50 fully connected nodes to combine the extracted structural features.
- Dense Output Layer: A single node acting as a linear regressor to output the continuous value forecast for time step $t$.

In [7]:
# Import the model architecture from the models directory
from models.cnn import build_cnn_model

# Configuration variables
n_steps = 3
n_features = 1

# Instantiate and build the model
cnn_model = build_cnn_model(n_steps, n_features)

# Display the structural summary and parameter counts
cnn_model.summary()

c:\Users\Lanie\Documents\Code\Keras-Time-Series-Architectures\.venv_tf\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 2, 64)          │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 50)             │         3,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,493 (13.64 KB)

 Trainable params: 3,493 (13.64 KB)

 Non-trainable params: 0 (0.00 B)

# 6. Long Short-Term Memory (LSTMs) and its Variations

### 6.1 Overview

While standard RNNs possess sequential memory, they fail to retain information across long lookback windows due to the vanishing gradient problem. Long Short-Term Memory (LSTM) networks solve this by introducing a specialized internal architecture centered around a cell state ($C_t$), which acts as a long-term memory conveyor belt. 

The cell state is regulated by three unique, trainable "gates" that control the flow of information:
- Forget Gate: Decides what information from historical time steps should be discarded or kept.
- Input Gate: Decides which new pieces of information from the current time step should be added to the cell state.
- Output Gate: Determines what the next hidden state ($h_t$)—and ultimately the prediction—should be, based on the updated cell state.

6.2 Architectural Variations & Implementations

We will implement three distinct structural variations of the LSTM in our framework.

**Variation A: Vanilla LSTM**

A Vanilla LSTM is the standard, foundational architecture consisting of a single hidden LSTM layer followed immediately by a dense, fully connected output layer.Tensor Transformations:Input Tensor: (Samples, Time Steps, Features)The Sequential Output: By default, Keras evaluates the entire sequence but only outputs a 2D vector of the final hidden state calculated at the very last time step.$$\text{Input Shape: } (\text{Samples}, 3, 1) \xrightarrow{\text{LSTM(units=50)}} \text{Output Shape: } (\text{Samples}, 50)$$

**Variation B: Stacked LSTM**

A Stacked LSTM deepens the network by chaining multiple hidden LSTM layers together. This allows the network to build a hierarchical representation of temporal patterns: early layers capture raw, low-level oscillations, while deeper layers interpret broader trends or multi-seasonal signals.The Crucial Parameter (return_sequences=True):To stack an LSTM layer directly on top of another, the preceding layer must output its entire history of hidden states across all time steps, not just the final one. This preserves the 3D tensor shape required by the next LSTM layer.

$$
\begin{aligned}
\text{Layer 1: Simple Output}
&\rightarrow (\text{Samples}, 50)
&&\mathbf{[Incompatible\ with\ Layer\ 2]} \\[8pt]
\text{Layer 1: } \texttt{return\_sequences=True}
&\rightarrow (\text{Samples}, 3, 50)
&&\mathbf{[Compatible\ with\ Layer\ 2]}
\end{aligned}
$$

**Variation C: Bidirectional LSTM**

In standard time-series modeling, sequences are evaluated chronologically from past to present. A Bidirectional LSTM creates a dual-pathway architecture. It splits the input sequence into two independent hidden layers:
1. Forward Pass: Evaluates the window chronologically ($t-3 \rightarrow t-2 \rightarrow t-1$).
2. Backward Pass: Evaluates the window in reverse chronological order ($t-1 \rightarrow t-2 \rightarrow t-3$).

Evaluating a lookback window backward can provide additional contextual boundaries, highlighting sudden changes or recent trends from a complementary perspective before mapping the features to the output layer.Tensor Transformations:The bidirectional wrapper runs two independent hidden layers of size $N$ simultaneously. It concatenates their final hidden outputs, doubling the feature vector size before passing it to the output layer.$$\text{Input Shape: } (\text{Samples}, 3, 1) \xrightarrow{\text{Bidirectional(LSTM(units=50))}} \text{Output Shape: } (\text{Samples}, 100)$$

In [9]:
# Import the LSTM variations from the models directory
from models.lstm import build_vanilla_lstm, build_stacked_lstm, build_bidirectional_lstm

# Configuration variables
n_steps = 3
n_features = 1

# Instantiate the configurations
vanilla_model = build_vanilla_lstm(n_steps, n_features, units=50)
stacked_model = build_stacked_lstm(n_steps, n_features, units=50)
bidirec_model = build_bidirectional_lstm(n_steps, n_features, units=50)

# Display a structural summary example
stacked_model.summary()

c:\Users\Lanie\Documents\Code\Keras-Time-Series-Architectures\.venv_tf\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 3, 50)          │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,651 (119.73 KB)

 Trainable params: 30,651 (119.73 KB)

 Non-trainable params: 0 (0.00 B)